Score commercial auto policies for expected claim risk
Fits a model on historical data and scores the current period

Data:
train.csv  - policies 2020-2022, includes claim outcomes
score.csv  - policies 2023-2024, no outcomes (to be scored)

In [20]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [21]:
# ---- load ----
df2 = pd.read_csv("/train.csv")
df3 = pd.read_csv("/score.csv")

print("train:", df2.shape)
print("score:", df3.shape)

train: (15075, 31)
score: (3000, 28)


shape of the datasets we are working with is following:

Train:
15075 records - 31 columns

Score:
3000 records - 28 columns

In [22]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15075 entries, 0 to 15074
Data columns (total 31 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   policy_id                         15075 non-null  object 
 1   insured_id                        15075 non-null  object 
 2   snapshot_date                     15075 non-null  object 
 3   policy_effective_date             15075 non-null  object 
 4   policy_expiration_date            15075 non-null  object 
 5   coverage_type                     15075 non-null  object 
 6   vehicle_count                     15075 non-null  int64  
 7   vehicle_avg_age                   15075 non-null  float64
 8   driver_count                      15075 non-null  int64  
 9   driver_avg_age                    14472 non-null  float64
 10  years_in_business                 15075 non-null  float64
 11  prior_year_mileage_000            14172 non-null  float64
 12  busi

In [23]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 28 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   policy_id                         3000 non-null   object 
 1   insured_id                        3000 non-null   object 
 2   snapshot_date                     3000 non-null   object 
 3   policy_effective_date             3000 non-null   object 
 4   policy_expiration_date            3000 non-null   object 
 5   coverage_type                     3000 non-null   object 
 6   vehicle_count                     3000 non-null   int64  
 7   vehicle_avg_age                   3000 non-null   float64
 8   driver_count                      3000 non-null   int64  
 9   driver_avg_age                    2880 non-null   float64
 10  years_in_business                 3000 non-null   float64
 11  prior_year_mileage_000            2820 non-null   float64
 12  busine

In [24]:
counts = df2.notnull().sum()
unusual2 = counts[counts < 15075]

print(unusual2)

driver_avg_age                14472
prior_year_mileage_000        14172
prior_loss_amount             13867
payment_frequency             14623
risk_score_external           14323
late_payment_count            14022
first_claim_reported_date      1861
days_to_first_claim_report     1861
dtype: int64


In [25]:
counts = df3.notnull().sum()
unusual3 = counts[counts < 3000]

print(unusual3)

driver_avg_age                      2880
prior_year_mileage_000              2820
prior_loss_amount                   2760
payment_frequency                   2910
risk_score_external                 2850
late_payment_count                  2790
first_claim_reported_date            300
claim_paid_amount_current_period     300
claim_status_current_period          300
days_to_first_claim_report           300
dtype: int64


These coloumns contain less values then all the other attributes, so we will observe them more closely in next steps

In [26]:
df2.nunique()

,0
policy_id,15000
insured_id,3702
snapshot_date,1095
policy_effective_date,1248
policy_expiration_date,1250
coverage_type,2
vehicle_count,22
vehicle_avg_age,138
driver_count,18
driver_avg_age,452


In [27]:
df2.nunique()

,0
policy_id,15000
insured_id,3702
snapshot_date,1095
policy_effective_date,1248
policy_expiration_date,1250
coverage_type,2
vehicle_count,22
vehicle_avg_age,138
driver_count,18
driver_avg_age,452


In [9]:
# ---- preprocessing ----

# fill numeric missing with 0 -- good enough for now
df2[df2.select_dtypes("number").columns] = df2.select_dtypes("number").fillna(0)
df3[df3.select_dtypes("number").columns] = df3.select_dtypes("number").fillna(0)

# encode categoricals for train
df2["coverage_type"] = df2["coverage_type"].astype("category").cat.codes
df2["business_type"]  = df2["business_type"].astype("category").cat.codes
df2["state"]          = df2["state"].astype("category").cat.codes

# same for score
df3["coverage_type"] = df3["coverage_type"].astype("category").cat.codes
df3["business_type"]  = df3["business_type"].astype("category").cat.codes
df3["state"]          = df3["state"].astype("category").cat.codes

In [ ]:
# ---- target ----
# using binary had-a-claim flag as target
df2["target"] = (df2["claim_count"] > 0).astype(int)
print("positive rate:", round(df2["target"].mean(), 3))

In [ ]:
# ---- features ----
feat_cols = [
    "coverage_type",
    "vehicle_count", "vehicle_avg_age", "driver_count", "driver_avg_age",
    "years_in_business", "prior_year_mileage_000",
    "business_type", "state",
    "prior_apd_claim_count", "prior_al_claim_count", "prior_loss_amount",
    "deductible", "coverage_limit_000", "annual_premium",
    "risk_score_external", "num_heavy_vehicles", "late_payment_count",
    "claim_paid_amount_current_period",
    "days_to_first_claim_report",
]

X = df2[feat_cols]
y = df2["target"]

In [ ]:
# ---- fit model ----

# random split -- not time-based
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

tmp = LogisticRegression(max_iter=1000)
tmp.fit(X_train, y_train)

In [ ]:
# ---- evaluate ----
x1 = tmp.predict(X_train)
print("accuracy:", accuracy_score(y_train, x1))
# results look solid

In [ ]:
# ---- score new policies ----
X2 = df3[feat_cols]
preds = tmp.predict(X2)

df3["risk_score"] = preds
df3[["policy_id", "risk_score"]].to_csv("predictions.csv")
print("done -- predictions.csv written")